# Stag Hunt — Decaying ε-greedy vs UCB

**Strand 2 — Repeated games with UCB.**

The cleanest demonstration of the headline mechanism. Two agents repeatedly
play a Stag Hunt with payoffs `STAG=5, HARE=1, SUCKER=0, TEMPTATION=3`,
averaged across `NUM_TRIALS` independent trials. We compare a decaying-ε-greedy
pair against a UCB pair and plot the probability of choosing Stag over time.

Classes live in `src/stag_hunt/`.

## Imports

In [ ]:
import sys, pathlib

_repo_root = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "requirements.txt").exists()
)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import numpy as np
import matplotlib.pyplot as plt

from src.stag_hunt.env import (
    STAG, HARE, STAG_PAYOFF, HARE_PAYOFF,
    StagHuntGame,
)
from src.stag_hunt.agents import DecayingEpsilonGreedyAgent, UCBAgent

## Parameters

In [ ]:
NUM_TRIALS = 200      # Number of independent experiments to average
NUM_STEPS = 10000     # Steps per experiment

## Simulation

In [ ]:
print(f"Running simulation: {NUM_TRIALS} trials, {NUM_STEPS} steps each...")
game = StagHuntGame()

# Track percentage of STAG actions over time across all trials
# Axis 0: Time, Axis 1: Agent Pair Type (0=E-Greedy, 1=UCB)
stag_percentages = np.zeros((NUM_STEPS, 2))

for trial in range(NUM_TRIALS):
    # --- Pair 1: Decaying Epsilon Greedy ---
    agent1_eg = DecayingEpsilonGreedyAgent()
    agent2_eg = DecayingEpsilonGreedyAgent()

    # --- Pair 2: UCB ---
    agent1_ucb = UCBAgent(c=2.0)
    agent2_ucb = UCBAgent(c=2.0)

    for t in range(NUM_STEPS):
        # 1. Epsilon Greedy Interaction
        a1_eg = agent1_eg.select_action()
        a2_eg = agent2_eg.select_action()
        r1_eg, r2_eg = game.get_payoffs(a1_eg, a2_eg)
        agent1_eg.update(a1_eg, r1_eg)
        agent2_eg.update(a2_eg, r2_eg)

        # Record if they chose Stag (averaged for the pair)
        if a1_eg == STAG: stag_percentages[t, 0] += 1
        if a2_eg == STAG: stag_percentages[t, 0] += 1

        # 2. UCB Interaction
        a1_ucb = agent1_ucb.select_action()
        a2_ucb = agent2_ucb.select_action()
        r1_ucb, r2_ucb = game.get_payoffs(a1_ucb, a2_ucb)
        agent1_ucb.update(a1_ucb, r1_ucb)
        agent2_ucb.update(a2_ucb, r2_ucb)

        # Record if they chose Stag
        if a1_ucb == STAG: stag_percentages[t, 1] += 1
        if a2_ucb == STAG: stag_percentages[t, 1] += 1

# Normalize counts to percentages (Trials * 2 Agents)
stag_percentages /= (NUM_TRIALS * 2)

## Plot

In [ ]:
plt.figure(figsize=(10, 6))

# Plot Epsilon Greedy
plt.plot(stag_percentages[:, 0], label=r'Decaying $\epsilon$-Greedy', color='red', alpha=0.8)

# Plot UCB
plt.plot(stag_percentages[:, 1], label='UCB (Upper Confidence Bound)', color='blue', alpha=0.8)

plt.title(f'Evolution of Cooperation (Stag Hunt)\nStag payoff={STAG_PAYOFF}, Hare payoff={HARE_PAYOFF}, {NUM_TRIALS} trials')
plt.xlabel('Time Step')
plt.ylabel('Probability of Choosing "Stag" (Cooperation)')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)

output_filename = '../results/stag_hunt_results.png'
plt.savefig(output_filename)
print(f"Simulation complete. Plot saved to {output_filename}")
plt.show()